# **COMP 2211 Introduction to Artificial Intelligence**
## Programming Assignment 2 — Roadside: Traffic Sign Recognition ##

---

## Introduction

HKUST is piloting autonomous delivery robots on campus roads. Before a robot may cross a junction, it must read the traffic signs around it. The only input available to the system is a small, often poorly lit crop from a low-cost camera: a 32×32 RGB color image (represented as a NumPy array).
In this assignment, you will design your own data splitting strategy (Part 1) and build this sign-recognition module using two distinct paradigms:

🛠️ Traditional Computer Vision (The "Hand-Crafted" Way; Parts 2-3)

Before deep learning, computer vision relied on manual feature engineering. You will implement these low-level image-processing operations yourself using pure NumPy (no high-level CV libraries allowed):
- Preprocessing & Filtering: Grayscale conversion, padding, and 2-D convolution (supporting stride and dilation).
- Feature Extraction: Sobel edge detection, pooling, and intensity histograms.
- Classification: Building a short numerical descriptor for each image and classifying them using a $K$-Nearest Neighbors (KNN) classifier.

🧠 Modern Deep Learning (The Convolutional Neural Network; Part 4)

Next, you will build the modern pipeline: a small Convolutional Neural Network (CNN) that is fed the raw pixels directly and learns its own optimal features.

Finally, you will compare both approaches on the same test set and analyze which traffic signs each paradigm confuses.

## Learning objectives

After completing this assignment, you should be able to:

1. Prevent Data Leakage: Perform group-aware train/val splits on burst camera data.
2. Build CV Operations from Scratch: Implement 2D convolution (supporting stride, padding, and dilation), pooling, and Sobel edge detection in pure NumPy.
3. Engineer Hand-Crafted Features: Extract spatial and intensity descriptors and classify them using a vectorized KNN classifier tuned on validation data.
4. Train and Understand CNNs: Build and train a Keras CNN on raw pixels, recognizing its learned filters as automated counterparts to your hand-crafted kernels.
5. Diagnose Performance: Analyze class confusion using confusion matrices.

## Dataset

The data comes from the **German Traffic Sign Recognition Benchmark (GTSRB)**. We use ten
of its classes, cropped to the bounding box of the sign and resized to 32×32 RGB:

| Label | Sign | | Label | Sign |
|---:|---|---|---:|---|
| 0 | Speed limit 30 | | 5 | Yield |
| 1 | Speed limit 50 | | 6 | Stop |
| 2 | Speed limit 70 | | 7 | No entry |
| 3 | Speed limit 80 | | 8 | Road work |
| 4 | Speed limit 100 | | 9 | Keep right |

Five of the ten classes are speed limit signs. They share the identical red circular border and differ only in the digits inside it, which at 32×32 is genuinely hard — that is deliberate, and it is what makes the confusion matrix at the end worth reading.

You are given two files:

| File | Contents |
|---|---|
| `signs_train.npz` | 5,400 training images, with labels **and a `track_id` column** |
| `signs_test.npz` | 500 test images with labels, 50 per class |

### Uploading the data

Upload **both** `.npz` files into your Colab session using the *Files* panel on the left,
then run the cells below. Files uploaded to Colab do not survive a runtime disconnect,
so you will need to upload them again after a long break.

## Setup

Run the next cell to import everything you need. **Do not edit it.** Every library permitted in this assignment is imported here; you do not need any additional imports, and several external libraries are explicitly forbidden in specific tasks.

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt
from keras import layers, Sequential, Input
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if __name__ == '__main__':
    SEED = 2211
    random.seed(SEED)
    np.random.seed(SEED)

In [ ]:
if __name__ == '__main__':
    train_file = np.load('signs_train.npz', allow_pickle=True)
    test_file = np.load('signs_test.npz', allow_pickle=True)

    X_pool = train_file['X']              # (5400, 32, 32, 3) uint8
    y_pool = train_file['y']              # (5400,) int
    track_pool = train_file['track_id']   # (5400,) int
    CLASS_NAMES = [str(name) for name in train_file['class_names']]

    X_test = test_file['X']               # (500, 32, 32, 3) uint8
    y_test = test_file['y']               # (500,) int

    print("training pool :", X_pool.shape, X_pool.dtype)
    print("test set      :", X_test.shape, X_test.dtype)
    print("classes       :", len(CLASS_NAMES))
    print("distinct tracks in the pool:", len(np.unique(track_pool)))

In [ ]:
if __name__ == '__main__':
    # One example of each class.
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for label, ax in enumerate(axes.ravel()):
        first = np.flatnonzero(y_pool == label)[0]
        ax.imshow(X_pool[first])
        ax.set_title(f"{label}: {CLASS_NAMES[label]}", fontsize=9)
        ax.axis('off')
    plt.suptitle("One sample per class")
    plt.tight_layout()
    plt.show()

## Self-check cells

Most tasks below are followed by a **self-check cell**. It runs your function on a few
tiny inputs whose answers are worked out by hand and printed for you, and tells you which
ones match.

Read them as a smoke alarm, not as the marking scheme. They cover a handful of cases; the
graders cover far more, and passing every self-check does not guarantee full marks. But a
self-check that *fails* is telling you something real, and you should fix it before moving
on; for example, a wrong `convolve2d` will quietly poison every number in Parts 2 and 3.

Do not edit these cells.

In [ ]:
def self_check(label, got, want, atol=1e-6):
    """Compare one result against a hand-computed expected value. Do not edit."""
    try:
        got = np.asarray(got, dtype=float)
    except Exception as error:
        print(f"  FAIL  {label}: returned something that is not numeric ({error})")
        return False
    want = np.asarray(want, dtype=float)
    if got.shape != want.shape:
        print(f"  FAIL  {label}: shape {got.shape}, expected {want.shape}")
        return False
    if not np.allclose(got, want, atol=atol):
        print(f"  FAIL  {label}: largest difference {np.abs(got - want).max():.3e}")
        print(f"        you returned {np.round(got, 4).tolist()}")
        print(f"        expected     {np.round(want, 4).tolist()}")
        return False
    print(f"  pass  {label}")
    return True


def self_check_raises(label, function, *args, **kwargs):
    """Check that a call rejects bad input with ValueError. Do not edit."""
    try:
        function(*args, **kwargs)
    except ValueError:
        print(f"  pass  {label}")
        return True
    except Exception as error:
        print(f"  FAIL  {label}: raised {type(error).__name__}, expected ValueError")
        return False
    print(f"  FAIL  {label}: no error raised, expected ValueError")
    return False


def self_check_report(results):
    """Summarize one task's self-check. Do not edit."""
    if all(results):
        print(f"  -> all {len(results)} checks passed")
    else:
        print(f"  -> {results.count(False)} of {len(results)} checks FAILED "
              f"(the graders test more cases than this)")


print("self-check helpers ready")

---
# Part 1 — Splitting the data (1 Task)

## Why this dataset needs care

GTSRB was not collected by photographing signs one at a time. A camera was mounted in a
car, and as the car approached each sign, it recorded **30 consecutive video frames**. All 30 frames depict the exact same physical sign, recorded fractions of a second apart, growing slightly larger in the field of view as the vehicle advances.

A group of 30 frames like this is called a **track**, and the `track_id` column tells you
which track each image belongs to. Run the cell below to see one.

In [ ]:
if __name__ == '__main__':
    example_track = track_pool[0]
    frames = np.flatnonzero(track_pool == example_track)

    fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
    for ax, frame in zip(axes, frames[::4]):
        ax.imshow(X_pool[frame])
        ax.axis('off')
    plt.suptitle(f"8 of the {len(frames)} frames in track {example_track} "
                 f"— all the same physical sign")
    plt.tight_layout()
    plt.show()

Those images are nearly identical. Now think about what happens if you split the pool into training and validation sets **at random, image by image**: frame 7 of a track lands in training and frame 8 of the same track lands in validation. Evaluating validation accuracy would then assess whether the model recognizes an image almost identical to one it memorized during training. The resulting validation accuracy would appear high while offering zero diagnostic value regarding generalization.

The result is a validation score that looks excellent and means nothing. You will measure
exactly how large that effect is in Part 3, once you have a classifier to measure it with.

The fix is to split **by track**: choose whole tracks for validation, and put *all* of
their frames on the validation side. Then no physical sign appears on both sides.

### Task 1 — Implement `split_by_track`

**Input**

- `track_ids`: `np.ndarray` of shape `(N,)`, the track each image belongs to
- `labels`: `np.ndarray` of shape `(N,)`, the class of each image
- `n_val_tracks`: `int`, how many tracks **per class** to hold out for validation
- `seed`: `int`, controls which tracks are chosen

**Output**

A tuple `(train_idx, val_idx)` of two 1-D integer arrays, each **sorted in ascending
order**, holding positions into the original arrays.

**Requirements**

- For **each class separately**, choose `n_val_tracks` distinct tracks at random. Every
  image belonging to a chosen track goes to `val_idx`; every other image goes to
  `train_idx`.
- Choosing per class keeps both splits class-balanced. Choosing `n_val_tracks` tracks
  globally would not preserve this balance.
- No track id may appear in both splits.
- Together, the two arrays must be mutually disjoint and contain every index from `0` to `N - 1` exactly once.
- Calling the function repeatedly with the same seed must return identical results.
- You must implement this logic from scratch using NumPy. Do not use `train_test_split`, `GroupShuffleSplit`, or any third-party splitting utilities.

**Hints**

- `np.random.default_rng(seed)` gives you a generator; `rng.choice(a, size=..., replace=False)`
  draws distinct elements from `a`.
- `np.unique` gives you the distinct tracks; combine it with a boolean mask on `labels`
  to get the tracks belonging to one class.
- `np.isin(track_ids, chosen_tracks)` builds the validation mask in one call, and
  `np.flatnonzero` turns a boolean mask into sorted indices.

In [ ]:
def split_by_track(track_ids, labels, n_val_tracks, seed):
    """Split image indices into training and validation sets, grouped by track.

    Returns (train_idx, val_idx), both 1-D int arrays sorted ascending.
    """
    raise NotImplementedError("You need to implement this function for Task 1.")


if __name__ == '__main__':
    train_idx, val_idx = split_by_track(track_pool, y_pool, n_val_tracks=4, seed=SEED)

    X_train, y_train = X_pool[train_idx], y_pool[train_idx]
    X_val, y_val = X_pool[val_idx], y_pool[val_idx]

    print("train:", X_train.shape[0], "images from",
          len(np.unique(track_pool[train_idx])), "tracks")
    print("val  :", X_val.shape[0], "images from",
          len(np.unique(track_pool[val_idx])), "tracks")

In [ ]:
if __name__ == '__main__':
    # Do not edit. If any assertion fails, revisit Task 1. 
    # Note that the grader will test more cases than this.
    shared = set(track_pool[train_idx]) & set(track_pool[val_idx])
    assert not shared, f"{len(shared)} tracks appear in BOTH splits"
    assert len(train_idx) + len(val_idx) == len(X_pool), "the splits do not cover the pool"
    assert len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0, "overlapping indices"
    assert np.array_equal(train_idx, np.sort(train_idx)), "train_idx must be sorted"
    assert np.array_equal(val_idx, np.sort(val_idx)), "val_idx must be sorted"

    again = split_by_track(track_pool, y_pool, n_val_tracks=4, seed=SEED)
    assert np.array_equal(again[0], train_idx), "not reproducible for a fixed seed"

    print("no shared tracks, splits cover the pool, result is reproducible")
    print("validation images per class:", np.bincount(y_val))

### Task 1 - Reflective questions

The following questions encourage deeper reflection on data splitting and experimental design. They are not graded, but answering them will strengthen your conceptual understanding:

**Question 1.** Suppose a classmate splits the pool with
`np.random.permutation` over the 5,400 image indices instead of by track, and reports a
validation accuracy of 0.97. Their code has no bug. Explain, in terms of what a track is,
why that number does not tell them how well their model would work on a sign it has never
seen.

**Question 2.** The validation set you just built holds out 4 tracks per class rather
than 4 tracks chosen at random from the whole pool. Give one concrete thing that could go
wrong if the 40 tracks were chosen at random from all 180, and say why per-class
selection avoids it.

*(You will measure the size of the effect in Question 1 for yourself in Part 3, after you
have a classifier.)*

**Answers**

*Question 1:* The 30 frames of one track are photographs of a single physical sign taken a fraction of a second apart, so they are near-duplicates of one another. An image-level random split puts some frames of a track in training and the rest in validation, so for almost every validation image there is an almost identical training image. Measuring accuracy on this split evaluates whether the model can recognise an image it has effectively already memorised, measuring memorisation rather than generalisation. The reported accuracy is high because the task has become trivial, not because the model has generalised well. On genuinely unseen signs—such as those in the test set—performance will drop significantly.

*Question 2:* With only 18 tracks per class, choosing 40 tracks at random from all 180
could easily take 7 or 8 tracks from one class and none from another. A class with no
validation tracks contributes nothing to the validation accuracy, so a model that
never predicts it would not be penalised, and a class that loses 8 of its 18 tracks is
left with under-represented training data. Holding out exactly 4 tracks per class
keeps the validation set balanced — 120 images per class — so that validation accuracy
weights every class equally and stays comparable to the balanced test set.

---
# Part 2 — A classical image-processing front end (6 Tasks)

Before neural networks, computer vision systems were built in two separate halves: a hand-written front end that turned an image into a short list of numbers, and a general-purpose classifier that worked on those numbers. In this part, you will write the front end. In Part 3, you will attach a classifier to it.

The front end you are going to build does six things:

```text
32x32x3 colour image
    └── to_grayscale  ──>  32x32 brightness image
                             ├──> intensity_histogram (16 bins) ────────┐
                             └──> convolve2d ──> edge_magnitude         │
                                                   └──> pool2d (8x8) ───┴──> build_feature_matrix (80-dim)
```

## Task 2 — Implement `to_grayscale`

A colour image stores three numbers per pixel. Much of a sign's identity is in its shape and its edges rather than its colour, and a grayscale representation requires one third of the memory, so the first step is to collapse the colour axis into a single brightness value.

The standard way to do this is **not** to average the three channels. Human vision is far
more sensitive to green than to blue, so the ITU-R BT.601 luma weights are used instead:

$$\text{gray} = 0.299\,R + 0.587\,G + 0.114\,B$$

**Input**

- `images`: `np.ndarray` with values in `[0, 255]`, either a single image of shape
  `(H, W, 3)` or a batch of shape `(N, H, W, 3)`

**Output**

- a float array with the colour axis removed — `(H, W)` or `(N, H, W)` — with values
  scaled to `[0, 1]`

**Requirements**

- Use the weights above, in that order (R, G, B).
- Divide by 255 so the result lies in `[0, 1]`.
- The same code must handle both a single image and a batch. Do not write a Python loop
  over images.

**Hint:** if `rgb` has shape `(..., 3)` and `w` has shape `(3,)`, then `rgb @ w`
contracts the last axis and works for any number of leading axes.

**Example**

```text
to_grayscale(np.array([[[255, 255, 255]]], dtype=np.uint8))  ->  array([[1.0]])
to_grayscale(np.array([[[0, 0, 0]]], dtype=np.uint8))        ->  array([[0.0]])
to_grayscale(np.array([[[255, 0, 0]]], dtype=np.uint8))      ->  array([[0.299]])
```

In [ ]:
def to_grayscale(images):
    """Convert RGB images in [0, 255] to grayscale in [0, 1] using BT.601 luma weights.

    Accepts (H, W, 3) or (N, H, W, 3) and removes the colour axis.
    """
    raise NotImplementedError("You need to implement this function for Task 2.")


if __name__ == '__main__':
    gray_train = to_grayscale(X_train)
    print("grayscale batch:", gray_train.shape, gray_train.dtype)
    print("value range    :", round(gray_train.min(), 4), "to", round(gray_train.max(), 4))

    fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
    for i, ax in enumerate(axes):
        ax.imshow(gray_train[i * 137], cmap='gray', vmin=0, vmax=1)
        ax.set_title(CLASS_NAMES[y_train[i * 137]], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == '__main__':
    print("Task 2 self-check")
    try:
        # A single 1x2 image: pure red, then white.
        probe = np.array([[[255, 0, 0], [255, 255, 255]]], dtype=np.uint8)
        results = [
            self_check("pure red and white", to_grayscale(probe), [[0.299, 1.0]]),
            self_check("all-black stays 0", to_grayscale(np.zeros((1, 2, 2, 3), np.uint8)),
                       np.zeros((1, 2, 2))),
            self_check("batch of 7 non-square images loses only the colour axis",
                       np.asarray(to_grayscale(np.zeros((7, 5, 3, 3), np.uint8))).shape,
                       [7, 5, 3]),
        ]
        self_check_report(results)
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 3 — Implement `convolve2d`

Convolution is the single operation that the rest of this part is built from, and it is
also the operation a CNN performs in Part 4. You slide a small array — the **kernel** —
across the image, and at each position you multiply the overlapping numbers elementwise
and add them up. That sum is one output pixel.

Three parameters control how the kernel slides:

- **stride** — how many pixels the kernel jumps between positions. A stride of 2 halves
  the output size in each direction.
- **padding** — whether the image is enlarged with a border of zeros first.
  With `'valid'` there is no padding, so the output is smaller than the input. With
  `'same'`, the image is padded by half the effective kernel size (see below), which for
  an odd-sized kernel and a stride of 1 makes the output the same size as the
  input.
- **dilation** — how far apart the kernel's own entries are spaced. With `dilation=1`,
  the kernel entries land on neighbouring pixels, as usual. With `dilation=2`, a gap of one pixel is left between entries, so that a 3x3 kernel reaches across a 5x5 receptive field while still using only 9 parameters. This enables a network to expand its receptive field without adding extra weights.



A `(kh, kw)` kernel at dilation `d` therefore covers an **effective** window of

$$kh_{\text{eff}} = d \times (kh - 1) + 1 \qquad kw_{\text{eff}} = d \times (kw - 1) + 1$$

and the output size in each dimension is the formula from class, with the effective
kernel size in place of the kernel size:

$$\text{out} = \left\lfloor \frac{\text{size} + 2 \times \text{padding} - \text{effective kernel size}}{\text{stride}} \right\rfloor + 1$$

For `'same'` padding, the pad is `(kh_eff - 1) // 2` rows top and bottom and
`(kw_eff - 1) // 2` columns left and right. Note that at `dilation=1` this is exactly
`(kh - 1) // 2`, so nothing changes for ordinary convolution.

**Input**

- `image`: 2-D `np.ndarray` of shape `(H, W)`
- `kernel`: 2-D `np.ndarray` of shape `(kh, kw)`
- `stride`: `int`, default `1`
- `padding`: either `'valid'` or `'same'`, default `'valid'`
- `dilation`: `int`, default `1`

**Output**

- a 2-D float array whose shape follows the formula above

**Requirements**

- **Do not flip the kernel.** You may assume the kernel is already flipped. This function computes cross-correlation, which is what Keras `Conv2D` also does.
- Pad with **zeros** only.
- `dilation=1` must behave exactly like ordinary convolution.
- Raise a `ValueError` if `padding` is neither `'valid'` nor `'same'`.
- You may not use `scipy.signal.convolve2d`, `scipy.signal.correlate2d`, `np.convolve`,
  or any Keras/TensorFlow convolution. `np.pad` is allowed and recommended.
- **Your function must be vectorised.** Part of the mark for this task comes from
  convolving a 320×320 image with a 17×17 kernel in under 4 seconds. The loop suggested
  below does it in a fraction of a second; a loop over every output pixel takes about
  nine.

**Hints**

- A double loop over the `kh × kw` *kernel* positions is enough, and is much faster than
  a double loop over the `out_h × out_w` output pixels — the kernel has 9 entries, the
  output has hundreds. This is exactly the structure used in the exam solution.
- For kernel entry `(i, j)`, the input rows it touches are
  `np.arange(0, out_h * stride, stride) + i * dilation`, and similarly for columns. The
  `* dilation` is the *only* place dilation enters the loop.
- `padded[np.ix_(rows, cols)]` selects the whole rectangular block those indices describe.

**Example**

```text
image  = [[1, 2, 3],        kernel = [[1, 0],
          [4, 5, 6],                  [0, 1]]
          [7, 8, 9]]

convolve2d(image, kernel, stride=1, padding='valid')
    ->  [[ 6,  8],          # 1*1 + 5*1 = 6,  2*1 + 6*1 = 8
         [12, 14]]          # 4*1 + 8*1 = 12, 5*1 + 9*1 = 14
```

And the same kernel with `dilation=2`, which spreads its two taps three pixels apart so
they no longer both fit in a 3×3 image — the effective window is 3×3, giving a 1×1 output:

```text
convolve2d(image, kernel, stride=1, padding='valid', dilation=2)
    ->  [[10]]              # 1*1 + 9*1 = 10   (corners of the 3x3 window)
```

In [ ]:
def convolve2d(image, kernel, stride=1, padding='valid', dilation=1):
    """Slide `kernel` over `image` and return the cross-correlation response.

    The kernel is NOT flipped. padding is 'valid' (none) or 'same' (zero padding of
    half the effective kernel size). dilation spaces the kernel entries out, so a
    (kh, kw) kernel covers dilation * (kh - 1) + 1 by dilation * (kw - 1) + 1 pixels.
    """
    raise NotImplementedError("You need to implement this function for Task 3.")


if __name__ == '__main__':
    demo_image = np.array([[1.0, 2.0, 3.0],
                           [4.0, 5.0, 6.0],
                           [7.0, 8.0, 9.0]])
    demo_kernel = np.array([[1.0, 0.0],
                            [0.0, 1.0]])
    print("worked example:")
    print(convolve2d(demo_image, demo_kernel, stride=1, padding='valid'))
    print("expected      : [[ 6.  8.]\n                 [12. 14.]]")
    print()
    print("same kernel, dilation 2:",
          convolve2d(demo_image, demo_kernel, stride=1, padding='valid', dilation=2))
    print("expected               : [[10.]]")

    sample = to_grayscale(X_train[42])
    box_blur = np.ones((3, 3)) / 9.0
    sharpen = np.array([[0.0, -1.0, 0.0],
                        [-1.0, 5.0, -1.0],
                        [0.0, -1.0, 0.0]])

    print()
    print("32x32 image, 3x3 kernel, stride 1, 'valid' ->",
          convolve2d(sample, box_blur, 1, 'valid').shape)
    print("32x32 image, 3x3 kernel, stride 1, 'same'  ->",
          convolve2d(sample, box_blur, 1, 'same').shape)
    print("32x32 image, 5x5 kernel, stride 2, 'valid' ->",
          convolve2d(sample, np.ones((5, 5)), 2, 'valid').shape)
    print("32x32 image, 3x3 kernel, dilation 3, 'valid' ->",
          convolve2d(sample, box_blur, 1, 'valid', 3).shape)

    fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
    for ax, (title, result) in zip(axes, [
        ("original", sample),
        ("box blur", convolve2d(sample, box_blur, 1, 'same')),
        ("sharpen", convolve2d(sample, sharpen, 1, 'same')),
        ("blur, dilation 3", convolve2d(sample, box_blur, 1, 'same', 3)),
    ]):
        ax.imshow(result, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == '__main__':
    print("Task 3 self-check")
    try:
        ramp = np.arange(16, dtype=float).reshape(4, 4)
        diff = np.array([[1.0, 0.0], [0.0, -1.0]])
        ones2, ones3 = np.ones((2, 2)), np.ones((3, 3))
        results = [
            self_check("2x2 kernel, valid, stride 1", convolve2d(ramp, diff),
                       np.full((3, 3), -5.0)),
            self_check("the same, stride 2", convolve2d(ramp, diff, stride=2),
                       np.full((2, 2), -5.0)),
            self_check("3x3 kernel, 'same', keeps the shape",
                       np.asarray(convolve2d(ramp, ones3, padding='same')).shape, [4, 4]),
            self_check("corner of that 'same' result is a zero-padded sum",
                       convolve2d(ramp, ones3, padding='same')[0, 0], 10.0),
            self_check("2x2 kernel at dilation 2 covers a 3x3 footprint",
                       convolve2d(ramp, ones2, dilation=2),
                       [[20.0, 24.0], [36.0, 40.0]]),
            self_check("dilation 1 is ordinary convolution",
                       convolve2d(ramp, ones2, dilation=1), convolve2d(ramp, ones2)),
            self_check_raises("an unknown padding is rejected",
                              convolve2d, ramp, diff, 1, 'reflect'),
        ]
        self_check_report(results)
        print("  if the first check gives +5 everywhere, you flipped the kernel")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 4 — Implement `edge_magnitude`

A traffic sign is mostly defined by its outline and the shapes painted on it, so the most
useful thing to measure is where the brightness *changes*. The **Sobel** kernels do this,
one for each direction:

```text
SOBEL_X = [[-1,  0,  1],        SOBEL_Y = [[-1, -2, -1],
           [-2,  0,  2],                   [ 0,  0,  0],
           [-1,  0,  1]]                   [ 1,  2,  1]]
```

`SOBEL_X` detects vertical edges (brightness changing as you move left to right) and
`SOBEL_Y` detects horizontal edges. Convolving with each gives two gradient images
`gx` and `gy`, and combining them gives the **edge magnitude**, which is large wherever
there is a strong edge in *any* direction:

$$\text{magnitude} = \sqrt{g_x^2 + g_y^2}$$

**Input**

- `gray`: 2-D `np.ndarray`, a single grayscale image of shape `(H, W)`

**Output**

- a 2-D float array of shape `(H, W)`

**Requirements**

- Apply both kernels with `stride=1` and `padding='same'`, so the output keeps the input
  shape.
- Call **your own** `convolve2d` from Task 3.
- Use the two kernel constants `SOBEL_X` and `SOBEL_Y` defined for you in the cell below.

**Note.** Because `'same'` padding surrounds the image with zeros, the outermost row and
column will always show a bright artificial edge — the padding looks like a sudden drop
to black. That is expected, and it affects every image equally.

In [ ]:
SOBEL_X = np.array([[-1.0, 0.0, 1.0],
                    [-2.0, 0.0, 2.0],
                    [-1.0, 0.0, 1.0]])
SOBEL_Y = np.array([[-1.0, -2.0, -1.0],
                    [0.0, 0.0, 0.0],
                    [1.0, 2.0, 1.0]])


def edge_magnitude(gray):
    """Return the Sobel gradient magnitude of a single grayscale image, same shape."""
    raise NotImplementedError("You need to implement this function for Task 4.")


if __name__ == '__main__':
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i in range(5):
        sample_gray = to_grayscale(X_train[i * 211])
        axes[0, i].imshow(sample_gray, cmap='gray', vmin=0, vmax=1)
        axes[0, i].set_title(CLASS_NAMES[y_train[i * 211]], fontsize=8)
        axes[1, i].imshow(edge_magnitude(sample_gray), cmap='magma')
        for ax in (axes[0, i], axes[1, i]):
            ax.axis('off')
    axes[0, 0].set_ylabel("grayscale")
    axes[1, 0].set_ylabel("edges")
    plt.suptitle("Grayscale (top) and Sobel edge magnitude (bottom)")
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == '__main__':
    print("Task 4 self-check")
    try:
        step = np.zeros((5, 5))
        step[:, 3:] = 1.0                      # a vertical edge between columns 2 and 3
        magnitude = np.asarray(edge_magnitude(step))
        results = [
            self_check("shape is preserved", magnitude.shape, [5, 5]),
            self_check("the middle row across the edge", magnitude[2],
                       [0.0, 0.0, 4.0, 4.0, 4.0]),
            self_check("a constant image has no interior edges",
                       edge_magnitude(np.full((5, 5), 0.3))[1:4, 1:4], np.zeros((3, 3))),
            self_check("a horizontal edge gives the same magnitude as a vertical one",
                       edge_magnitude(step.T)[2, 2], magnitude[2, 2]),
        ]
        self_check_report(results)
        print("  a zero middle row means only one of the two Sobel kernels is being applied")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 5 — Implement `intensity_histogram`

Edges tell you about shape. A **histogram** tells you about brightness: it counts how
many pixels fall into each brightness band, and throws away *where* those pixels were.

That makes it a useful complement to the edge map — a "Stop" sign and a "Yield" sign have
quite different overall brightness distributions even though both are mostly red and
white.

**Input**

- `gray`: `np.ndarray` of any shape, expected to hold values in `[0, 1]`
- `n_bins`: `int`, how many bins to divide `[0, 1]` into

**Output**

- a 1-D float array of length `n_bins` that **sums to 1**

**Requirements**

- Bin `b` covers `[b / n_bins, (b + 1) / n_bins)`, except the **last** bin, which also
  includes the value `1.0` exactly.
- Values below 0 are counted in the first bin and values above 1 in the last bin, so the
  function never crashes on out-of-range input.
- Normalise by the total number of pixels, so that the result is a distribution rather than
  raw counts. This makes histograms of different-sized images comparable.
- The input may have any shape; flatten it first.

**Hints**

- `np.clip` handles the out-of-range rule in one call.
- Multiplying a value in `[0, 1]` by `n_bins` and taking the integer part gives its bin —
  but `1.0 * n_bins` is `n_bins`, one past the end, so clamp the result to `n_bins - 1`.
- `np.bincount(indices, minlength=n_bins)` counts the bins, including empty ones.

**Example**

```text
intensity_histogram(np.array([0.0, 0.24, 0.26, 0.5, 0.99, 1.0]), n_bins=4)
    ->  array([0.333..., 0.166..., 0.166..., 0.333...])       # counts [2, 1, 1, 2] / 6
```

In [ ]:
def intensity_histogram(gray, n_bins):
    """Histogram a grayscale image over [0, 1], normalised to sum to 1."""
    raise NotImplementedError("You need to implement this function for Task 5.")


if __name__ == '__main__':
    example = intensity_histogram(np.array([0.0, 0.24, 0.26, 0.5, 0.99, 1.0]), 4)
    print("worked example:", np.round(example, 4))
    print("expected      : [0.3333 0.1667 0.1667 0.3333]")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, label in zip(axes, [6, 5, 9]):
        first = np.flatnonzero(y_train == label)[0]
        hist = intensity_histogram(to_grayscale(X_train[first]), 16)
        ax.bar(np.arange(16), hist)
        ax.set_title(CLASS_NAMES[label], fontsize=9)
        ax.set_xlabel("brightness bin")
    axes[0].set_ylabel("fraction of pixels")
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == '__main__':
    print("Task 5 self-check")
    try:
        probe = np.array([[0.0, 0.1], [0.6, 1.0]])
        results = [
            self_check("four bins over a fixed [0, 1] range",
                       intensity_histogram(probe, 4), [0.5, 0.0, 0.25, 0.25]),
            self_check("the value 1.0 belongs to the last bin",
                       intensity_histogram(np.ones((2, 2)), 5), [0, 0, 0, 0, 1]),
            self_check("length matches n_bins",
                       len(intensity_histogram(probe, 11)), 11),
            self_check("it sums to 1", intensity_histogram(probe, 7).sum(), 1.0),
            self_check("values outside [0, 1] are clipped into the end bins",
                       intensity_histogram(np.array([[-2.0, 3.0]]), 4), [0.5, 0.0, 0.0, 0.5]),
        ]
        self_check_report(results)
        print("  a histogram summing to 4 instead of 1 means you returned raw counts")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 6 — Implement `pool2d`

The edge map from Task 4 is 32×32 = 1,024 numbers per image. That is far too many to
hand to a classifier (especially not in a deep learning context), and most of them say almost the same thing as their neighbours.
**Pooling** shrinks the map by replacing each small window with a single number.

This is the same operation as `keras.layers.MaxPooling2D`, which you will use without
thinking about it in Part 4 — so implement it once here and you will know exactly what
that layer does.

Two modes:

- `'max'` — keep the largest value in the window. This answers "is there a strong edge
  anywhere in this region?" and is what CNNs almost always use.
- `'mean'` — average the window. This answers "how much edge energy is in this region on
  average?" and is what Task 7 will use to build its descriptor.

Pooling never pads, so the output size is the `'valid'` case of the Task 3 formula, with
the pool size in place of the kernel size:

$$\text{out} = \left\lfloor \frac{\text{size} - \text{pool size}}{\text{stride}} \right\rfloor + 1$$

A partial window at the right or bottom edge is simply dropped.

**Input**

- `feature_map`: 2-D `np.ndarray` of shape `(H, W)`
- `pool_size`: `int`, the window is `pool_size × pool_size`
- `stride`: `int` or `None`, default `None`
- `mode`: `'max'` or `'mean'`, default `'max'`

**Output**

- a 2-D pooling result whose shape follows the formula above

**Requirements**

- `stride=None` means `stride = pool_size`, which gives non-overlapping windows.
  That is the default `MaxPooling2D` uses too. Do not silently use a stride of 1.
- Raise a `ValueError` if `mode` is anything other than `'max'` or `'mean'`, and if the
  window does not fit inside the input at all.
- Do not modify `feature_map` in place.
- You may not use `keras.layers.MaxPooling2D`, `skimage.measure.block_reduce`, or any
  library pooling function.

**Hints**

- The same trick as Task 3 works here: instead of looping over output pixels, loop over
  the `pool_size × pool_size` positions *inside* the window. For window position
  `(i, j)`, the values it contributes are
  `feature_map[np.ix_(np.arange(out_h) * stride + i, np.arange(out_w) * stride + j)]`.
- Stack those `pool_size²` 1-D arrays with `np.stack` and reduce along axis 0 with `.max()`
  or `.mean()`.

**Example**

```text
feature_map = [[ 0,  1,  2,  3],
               [ 4,  5,  6,  7],
               [ 8,  9, 10, 11],
               [12, 13, 14, 15]]

pool2d(feature_map, 2)                  ->  [[ 5,  7],      # max of each 2x2 block
                                             [13, 15]]

pool2d(feature_map, 2, mode='mean')     ->  [[ 2.5,  4.5],  # mean of each 2x2 block
                                             [10.5, 12.5]]

pool2d(feature_map, 2, stride=1)        ->  [[ 5,  6,  7],  # overlapping windows
                                             [ 9, 10, 11],
                                             [13, 14, 15]]
```

In [ ]:
def pool2d(feature_map, pool_size, stride=None, mode='max'):
    """Downsample a 2-D map by reducing each pool_size x pool_size window to one value.

    Stride defaults to pool_size (non-overlapping windows). Supported modes are 'max' and 'mean'.
    Output spatial dimensions follow out = (size - pool_size) // stride + 1.
    """
    raise NotImplementedError("You need to implement this function for Task 6.")


if __name__ == '__main__':
    demo = np.arange(16.0).reshape(4, 4)
    print("worked example, max :", pool2d(demo, 2).tolist())
    print("expected            : [[5.0, 7.0], [13.0, 15.0]]")
    print("worked example, mean:", pool2d(demo, 2, mode='mean').tolist())
    print("expected            : [[2.5, 4.5], [10.5, 12.5]]")
    print("overlapping, stride 1, shape:", pool2d(demo, 2, stride=1).shape,
          "(expected (3, 3))")

    sample_edges = edge_magnitude(to_grayscale(X_train[42]))
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
    for ax, (title, result) in zip(axes, [
        ("edge map 32x32", sample_edges),
        ("max pool 4x4", pool2d(sample_edges, 4, mode='max')),
        ("mean pool 4x4", pool2d(sample_edges, 4, mode='mean')),
    ]):
        ax.imshow(result, cmap='magma')
        ax.set_title(f"{title}  {result.shape}", fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == '__main__':
    print("Task 6 self-check")
    try:
        ramp = np.arange(16, dtype=float).reshape(4, 4)
        results = [
            self_check("2x2 max pooling", pool2d(ramp, 2), [[5.0, 7.0], [13.0, 15.0]]),
            self_check("2x2 mean pooling", pool2d(ramp, 2, mode='mean'),
                       [[2.5, 4.5], [10.5, 12.5]]),
            self_check("stride defaults to pool_size, not to 1",
                       np.asarray(pool2d(ramp, 2)).shape, [2, 2]),
            self_check("an explicit stride of 1 overlaps the windows",
                       pool2d(ramp, 2, stride=1),
                       [[5.0, 6.0, 7.0], [9.0, 10.0, 11.0], [13.0, 14.0, 15.0]]),
            self_check("a partial window at the edge is dropped",
                       np.asarray(pool2d(np.arange(25, dtype=float).reshape(5, 5), 2)).shape,
                       [2, 2]),
            self_check_raises("an unknown mode is rejected", pool2d, ramp, 2, None, 'median'),
            self_check_raises("a window larger than the input is rejected", pool2d, ramp, 9),
        ]
        self_check_report(results)
        print("  a 3x3 result on the 5x5 input means partial windows are being kept")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 7 — Implement `build_feature_matrix`

Now put the pieces together. Every image becomes one row of numbers, and the whole
dataset becomes a matrix that a classifier can consume.

The descriptor has two halves:

1. **What brightnesses are present** — the `n_bins`-bin intensity histogram from Task 5.
   This has no idea *where* anything is.
2. **Where the edges are** — the Sobel edge map from Task 4, mean-pooled with your Task 6
   `pool2d` using a window and stride of `block_size`. With a 32×32 image and
   `block_size=4`, this gives an 8×8 grid of numbers, each saying "how much edge energy is
   in this part of the image". This is coarse, but unlike the histogram, it keeps spatial
   information.

With the defaults, the descriptor is `16 + 64 = 80` numbers long.

Look at what you have just built, end to end: **convolve** with a fixed kernel, apply a
**non-linearity** (the square root of a sum of squares), **pool**, **flatten**, and hand
the result to a classifier. That is exactly the structure of the CNN you will build in Part 4.
The only difference is that here *you* choose the kernels, and there the network learns
them.

**Input**

- `images`: `np.ndarray` of shape `(N, H, W, 3)`, or a single `(H, W, 3)` image
- `n_bins`: `int`, default `16`
- `block_size`: `int`, default `4`

**Output**

- a 2-D float array of shape `(N, n_bins + (H // block_size) * (W // block_size))`

**Requirements**

- For each image: convert to grayscale, compute its histogram, compute its edge map,
  mean-pool the edge map, and concatenate the **histogram first, then pooled blocks**.
- The pooled values must be read in **row-major order** (left to right, then top to
  bottom) — this is what `.ravel()` does by default.
- A single image must produce a 2D matrix of shape $(1, D)$, not a 1-D array $(D,)$.
- Call your own functions from Tasks 2, 4, 5 and 6. Pool with
  `mode='mean'` — max pooling is measured further down and does *worse* here.

**Hints**

- Convert the whole batch to grayscale in one call; only the per-image steps need a loop.
- `pool2d(edges, block_size, stride=block_size, mode='mean')` is the whole pooling step.
- `np.concatenate([hist, blocks.ravel()])` builds one row.

In [ ]:
def build_feature_matrix(images, n_bins=16, block_size=4):
    """Turn a batch of RGB images into a hand-crafted feature matrix of shape (N, D)."""
    raise NotImplementedError("You need to implement this function for Task 7.")


if __name__ == '__main__':
    F_train = build_feature_matrix(X_train)
    F_val = build_feature_matrix(X_val)
    F_test = build_feature_matrix(X_test)

    print("train features:", F_train.shape)
    print("val features  :", F_val.shape)
    print("test features :", F_test.shape)
    print("histogram part of row 0 sums to:",
          round(F_train[0, :16].sum(), 10))
    assert F_train.shape[1] == 80, "expected 80 features per image"

In [ ]:
if __name__ == '__main__':
    print("Task 7 self-check")
    try:
        probe = np.broadcast_to(np.tile(np.arange(256).reshape(16, 16), (2, 2))[..., None], (3, 32, 32, 3))
        features = np.asarray(build_feature_matrix(probe))
        expected_features_10_20 = np.tile(np.array([0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.47124057, 0.46172439, 0.47737836, 0.49944476]), (3, 1))
        results = [
            self_check("default shape is (N, 16 + 8*8)", features.shape, [3, 80]),
            self_check("the histogram comes first and still sums to 1",
                       features[:, :16].sum(axis=1), np.ones(3)),
            self_check("a single image gives one row",
                       np.asarray(build_feature_matrix(probe[:1])).shape, [1, 80]),
            self_check("n_bins and block_size are both honoured",
                       np.asarray(build_feature_matrix(probe, n_bins=8, block_size=8)).shape,
                       [3, 8 + 16]),
            self_check("each row depends only on its own image",
                       build_feature_matrix(probe)[1], build_feature_matrix(probe[1:2])[0]),
            self_check("no NaN or inf", np.isfinite(features).all(), True),
            self_check("expected values check", np.allclose(features[:, 10:20], expected_features_10_20, atol=1e-5), True)
        ]
        self_check_report(results)
        print("  a shape of (3, 80) with a histogram that does not sum to 1 means the two "
              "halves are the wrong way round")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

---
# Part 3 — A K-Nearest-Neighbour baseline (2 Tasks)

You now have an 80-number description of every image. The next step is to classify those
descriptions — but first they need to be put on a common scale.

## Task 8 — Implement `standardize`

The 80 features are not comparable as they stand. The histogram entries are fractions
between 0 and 1 that sum to one, so each is typically around 0.06. The edge block averages
are Sobel magnitudes, which can be several times larger. Since K-Nearest Neighbours uses
Euclidean distance to measure the distance between samples, unscaled edge features would dominate the distance metric and completely suppress the contribution of histogram features.

The standard fix is to rescale each column to zero mean and unit standard deviation:

$$z = \frac{x - \mu}{\sigma}$$
**Critical Rule**: Both $\mu$ and $\sigma$ must be computed from the **training data
only**, and then applied unchanged to validation and test data. If you compute them over
everything, information about the test set — how bright it is, how much it varies — has
influenced your training pipeline, and your reported accuracy is no longer an honest
estimate of performance on data you have never seen. That is **data leakage**. In
production, the test data does not exist yet, so any statistic you could not have computed
at training time is a statistic you are not allowed to use.

**Input**

- `X_train`: `np.ndarray` of shape `(N, D)`
- `X_other`: `np.ndarray` of shape `(M, D)`

**Output**

- a tuple `(X_train_scaled, X_other_scaled)` with the same shapes as the inputs

**Requirements**

- Compute the mean and standard deviation **per column**, from `X_train` only.
- Apply those same statistics to both matrices.
- If a column's training standard deviation is exactly 0, dividing by 0 will produce `NaN` or `inf`. In this case, simply set that column to 0 in both outputs.
- Do not modify the input arrays in place.
- You may not use external library scalers, such as `sklearn.preprocessing.StandardScaler` (`numpy` is allowed).

**Hint:** `np.where(std == 0, 1.0, std)` gives you a divisor that is safe everywhere.

In [ ]:
def standardize(X_train, X_other):
    """Standardise both matrices using the column mean and std of X_train ONLY.

    Returns (X_train_scaled, X_other_scaled).
    """
    raise NotImplementedError("You need to implement this function for Task 8.")


if __name__ == '__main__':
    F_train_s, F_test_s = standardize(F_train, F_test)
    _, F_val_s = standardize(F_train, F_val)

    print("train column means (should be ~0):", np.round(F_train_s.mean(axis=0)[:4], 10))
    print("train column stds  (should be ~1):", np.round(F_train_s.std(axis=0)[:4], 10))
    print("test column means  (need NOT be 0):", np.round(F_test_s.mean(axis=0)[:4], 4))

In [ ]:
if __name__ == '__main__':
    print("Task 8 self-check")
    try:
        # Column 1 varies; column 2 is constant, which is the case that causes division by zero.
        A = np.array([[1.0, 10.0], [3.0, 10.0], [5.0, 10.0]])
        B = np.array([[7.0, 10.0]])
        A_scaled, B_scaled = standardize(A, B)
        results = [
            self_check("training column has zero mean", np.asarray(A_scaled).mean(axis=0),
                       [0.0, 0.0]),
            self_check("training column has unit std", np.asarray(A_scaled)[:, 0].std(), 1.0),
            self_check("the SAME statistics are applied to the second matrix",
                       B_scaled, [[2.449489, 0.0]]),
            self_check("a constant column produces no NaN or inf",
                       np.isfinite(np.asarray(B_scaled)).all(), True),
            self_check("the inputs are not modified in place", A, [[1.0, 10.0], [3.0, 10.0],
                                                                  [5.0, 10.0]]),
        ]
        self_check_report(results)
        print("  if the third check fails, you fitted the scaling on B as well — that is the "
              "leakage this task is about")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

## Task 9 — Implement `knn_predict`

K-Nearest Neighbours does not train at all. To classify a query image, it finds the `k`
training images whose features are closest to the query's, and returns whichever label
is most common among them.

The only difficulty is doing it *fast*. There are 5,400 training rows and 500 query
rows, so a Python loop over every pair does 2.7 million distance computations and takes
minutes. Expanding the squared Euclidean distance lets NumPy do all of them in one matrix
multiplication, avoiding the need to construct a massive temporary array of differences $a-b$:

$$\|a - b\|^2 = \|a\|^2 - 2\,a \cdot b + \|b\|^2$$

Applied to whole matrices, with broadcasting supplying the two outer terms, this gives
every pairwise squared distance at once.

**Input**

- `X_train`: `np.ndarray` of shape `(N, D)`
- `y_train`: `np.ndarray` of shape `(N,)`, integer labels
- `X_query`: `np.ndarray` of shape `(M, D)`
- `k`: `int`, the number of neighbours to consult

**Output**

- a 1-D integer array of length `M` holding the predicted label for each query row

**Requirements**

- Use Euclidean distance. You may compare **squared** distances throughout — the square
  root is a monotonic function and does not change which neighbours are nearest, so
  taking it is wasted work.
- If there is a tie in the majority vote, the class with the smallest label index should be chosen. (Using `np.bincount(...).argmax()` satisfies this directly, as `argmax` returns the first occurring maximum.)
- Your implementation must be vectorised: no Python loop over training rows, and no
  nested loop over pairs. A single loop over query rows to count votes is acceptable.
- You may not use `sklearn.neighbors.KNeighborsClassifier`.

**Hints**

- `(X_query ** 2).sum(axis=1)[:, np.newaxis]` has shape `(M, 1)` and
  `(X_train ** 2).sum(axis=1)[np.newaxis, :]` has shape `(1, N)`; adding them to
  `-2 * X_query @ X_train.T` broadcasts to the full `(M, N)` distance matrix.
- `np.argpartition(distances, kth=k-1, axis=1)[:, :k]` finds the `k` smallest entries per
  row without fully sorting, which is faster than `np.argsort`.
- `np.bincount(labels_of_neighbours, minlength=n_classes).argmax()` performs the vote.

**Example**

```text
X_train = [[0, 0], [0, 0.1], [5, 5], [5, 5.1], [5, 4.9]]
y_train = [0, 0, 1, 1, 1]

knn_predict(X_train, y_train, [[0, 0.05], [5, 5]], k=3)  ->  array([0, 1])
```

In [ ]:
def knn_predict(X_train, y_train, X_query, k):
    """Predict labels for X_query by majority vote over the k nearest training rows.

    Ties are broken by choosing the smallest label.
    """
    raise NotImplementedError("You need to implement this function for Task 9.")


if __name__ == '__main__':
    demo_X = np.array([[0.0, 0.0], [0.0, 0.1], [5.0, 5.0], [5.0, 5.1], [5.0, 4.9]])
    demo_y = np.array([0, 0, 1, 1, 1])
    print("worked example:", knn_predict(demo_X, demo_y, np.array([[0.0, 0.05],
                                                                  [5.0, 5.0]]), 3))
    print("expected      : [0 1]")

In [ ]:
if __name__ == '__main__':
    print("Task 9 self-check")
    try:
        # Two tight clusters: class 0 near the origin, class 1 near (5, 5).
        X_demo = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 5.0], [5.0, 6.0]])
        y_demo = np.array([0, 0, 1, 1])
        Q = np.array([[0.0, 0.2], [4.5, 5.5], [2.5, 2.5]])
        results = [
            self_check("k = 1", knn_predict(X_demo, y_demo, Q, 1), [0, 1, 0]),
            self_check("k = 3", knn_predict(X_demo, y_demo, Q, 3), [0, 1, 0]),
            self_check("one prediction per query row",
                       len(knn_predict(X_demo, y_demo, Q, 1)), 3),
            self_check("a tie resolves to the smaller label",
                       knn_predict(X_demo, y_demo, np.array([[2.5, 3.0]]), 4), [0]),
            self_check("labels come only from the training set",
                       np.isin(knn_predict(X_demo, y_demo, Q, 3), y_demo).all(), True),
        ]
        self_check_report(results)
        print("  [1, 0, 1] means you are taking the k FARTHEST rows")
    except Exception as error:
        print(f"  the self-check could not run: "
              f"{type(error).__name__}: {error}")
        print("  this normally means the function is still a TO DO stub")

Now, you can use your KNN classifier to predict the labels of the validation set you built in Part 1. This part won't contribute to your final mark, but it will give you a sense of how well your hand-crafted features work. Have fun :)

In [ ]:
if __name__ == '__main__':
    # Feel free to change the n_bins and block_size arguments to see how they affect accuracy.
    F_train = build_feature_matrix(X_train, n_bins=16, block_size=4)
    F_val = build_feature_matrix(X_val, n_bins=16, block_size=4)
    F_test = build_feature_matrix(X_test, n_bins=16, block_size=4)
    F_train_s, F_test_s = standardize(F_train, F_test)
    F_train_s, F_val_s = standardize(F_train, F_val)

    print("choosing k on the validation set:")
    for candidate_k in [1, 3, 5, 7, 9]:
        accuracy = np.mean(knn_predict(F_train_s, y_train, F_val_s, candidate_k) == y_val)
        print(f"  k = {candidate_k}: validation accuracy = {accuracy:.4f}")

### Was mean pooling the right choice?

Task 7 pools the edge map with `mode='mean'`. The obvious alternative is `mode='max'`,
which is what the CNN in Part 4 will use. The cell below rebuilds the whole descriptor
with max pooling and re-runs the same KNN, so you can see the answer rather than assume
it. Nothing here is graded — but the result is worth a moment's thought, because
"the CNN uses max pooling, so max pooling must be better" is exactly the kind of
reasoning this course wants you to stop doing.

In [ ]:
if __name__ == '__main__':
    def max_pooled_features(images, n_bins=16, block_size=4):
        descriptors = []
        for image in np.asarray(images):
            gray = to_grayscale(image)
            blocks = pool2d(edge_magnitude(gray), block_size, block_size, 'max')
            descriptors.append(
                np.concatenate([intensity_histogram(gray, n_bins), blocks.ravel()]))
        return np.stack(descriptors)

    M_train = max_pooled_features(X_train, n_bins=16, block_size=4) # change this to match your mean-pooled features if you like
    M_test = max_pooled_features(X_test, n_bins=16, block_size=4) # change this to match your mean-pooled features if you like
    M_train_s, M_test_s = standardize(M_train, M_test)

    BEST_K = 5 # change this based on your validation results if you like
    mean_pool_accuracy = np.mean(knn_predict(F_train_s, y_train, F_test_s, BEST_K) == y_test)
    max_pool_accuracy = np.mean(knn_predict(M_train_s, y_train, M_test_s, BEST_K) == y_test)

    print(f"KNN test accuracy, mean pooling: {mean_pool_accuracy:.4f}")
    print(f"KNN test accuracy, max pooling : {max_pool_accuracy:.4f}")
    print()
    print("Max pooling keeps only the strongest edge in each block and throws away how")
    print("much edge there was overall. For 32x32 signs that discards more than it gains.")
    print("A CNN can afford max pooling because it learns kernels whose strongest")
    print("response is already the informative one; a fixed Sobel kernel cannot.")

## What would have happened without Task 1

You now have a classifier, so you can measure the cost of splitting badly.

The cell below splits the same pool a second time, the obvious way: shuffle all 5,400
images and take the first 4,200 for training and the rest for validation. It keeps the
split sizes identical and changes only *how* images are assigned. Everything downstream —
features, standardisation, KNN — is the same code you just wrote.

Read the four numbers it prints before continuing.

In [ ]:
if __name__ == '__main__':
    # Staff demonstration. Do not edit.
    rng = np.random.default_rng(SEED)
    shuffled = rng.permutation(len(X_pool))
    naive_train_idx = shuffled[:len(train_idx)]
    naive_val_idx = shuffled[len(train_idx):]

    naive_F_train = build_feature_matrix(X_pool[naive_train_idx])
    naive_F_val = build_feature_matrix(X_pool[naive_val_idx])
    naive_train_s, naive_val_s = standardize(naive_F_train, naive_F_val)
    _, naive_test_s = standardize(naive_F_train, F_test)

    naive_val_acc = np.mean(
        knn_predict(naive_train_s, y_pool[naive_train_idx], naive_val_s, 1)
        == y_pool[naive_val_idx])
    naive_test_acc = np.mean(
        knn_predict(naive_train_s, y_pool[naive_train_idx], naive_test_s, 1) == y_test)

    grouped_val_acc = np.mean(knn_predict(F_train_s, y_train, F_val_s, 1) == y_val)
    grouped_test_acc = np.mean(knn_predict(F_train_s, y_train, F_test_s, 1) == y_test)

    shared_tracks = len(set(track_pool[naive_train_idx]) & set(track_pool[naive_val_idx]))

    print(f"tracks appearing on BOTH sides of the naive split: {shared_tracks} / 180")
    print()
    print(f"{'':22s}{'validation':>12s}{'test':>10s}{'gap':>10s}")
    print(f"{'split by track':22s}{grouped_val_acc:12.4f}{grouped_test_acc:10.4f}"
          f"{grouped_val_acc - grouped_test_acc:+10.4f}")
    print(f"{'split by image':22s}{naive_val_acc:12.4f}{naive_test_acc:10.4f}"
          f"{naive_val_acc - naive_test_acc:+10.4f}")

The naive split reports a validation accuracy in the high nineties. The true test
accuracy of that same model is roughly half of it.

Nothing about the model improved — the classifier is identical, and its performance on
genuinely unseen signs barely moved. What changed is that the validation set stopped being
a measurement and became a lookup: for almost every validation image there is a nearly
identical frame of the *same physical sign* sitting in the training set, and 1-NN finds it.

The split by track reports a validation accuracy within about a percentage point of the
test accuracy. It is much less flattering, and it is the only one of the two you could
have made a decision with.

**This is the general lesson.** Whenever your data has natural groups — frames from one
video, several photographs of one patient, multiple reviews by one author, repeated
measurements of one device — the split has to keep each group whole. Otherwise, your
validation score measures memorisation rather than generalisation.

---
# Part 4 — A convolutional neural network (2 Tasks)

The front end you wrote in Part 2 was designed by hand. Someone decided that intensity
histograms and block-averaged Sobel edges were the right things to measure, and those
80 numbers are all the classifier ever sees — anything they discard is gone.

A CNN inverts that. It is given the raw 32×32×3 pixels and *learns* what to measure
by learning the kernel values themselves. The `Conv2D` layer performs exactly the
operation you implemented in Task 3; the difference is that its kernels start random
and are adjusted by gradient descent.

## Task 10 — Implement `build_cnn`

Build and compile a CNN for 10-class classification. You have freedom in the architecture,
but you may only use these layers:

| Layer | Purpose |
|---|---|
| [`Input(shape=...)`](https://keras.io/api/layers/core_layers/input/) | declares the input shape; must be first |
| [`layers.Conv2D(filters, kernel_size, padding=, activation=)`](https://keras.io/api/layers/convolution_layers/convolution2d/) | learned convolution |
| [`layers.MaxPooling2D(pool_size)`](https://keras.io/api/layers/pooling_layers/max_pooling2d/) | reduces the spatial size, keeps the strongest response |
| [`layers.Flatten()`](https://keras.io/api/layers/reshaping_layers/flatten/) | turns the final feature maps into a vector |
| [`layers.Dense(units, activation=)`](https://keras.io/api/layers/core_layers/dense/) | fully connected layer |
| [`layers.Dropout(rate)`](https://keras.io/api/layers/regularization_layers/dropout/) | randomly zeroes activations during training, to reduce chances of overfitting |

**Input**

- `input_shape`: tuple, `(32, 32, 3)` for this dataset
- `n_classes`: `int`, `10` for this dataset

**Output**

- a **compiled** `keras.Sequential` model

**Requirements**

- The first layer must be `Input()`.
- Use at least **two** `Conv2D` layers and at least **one** `MaxPooling2D` layer.
- The last layer must be `Dense()` — one output per class,
  producing a probability distribution. This is the standard choice for multi-class
  classification. Choose your activation function wisely.
- Compile with `loss='sparse_categorical_crossentropy'`. Our labels are plain integers
  `0..9`, not one-hot vectors, which is what the *sparse* version expects. We recommend using accuracy as a metric and the `'adam'` optimizer.
- Keep the model under **500,000 trainable parameters**. A larger model will not train
  meaningfully faster on this data and is more likely to overfit on 4,200 images.

**Hint:** the reference solution uses around 150,000 trainable parameters. You do not have to match it — anything that satisfies the requirements above and trains well is fine.

In [ ]:
def build_cnn(input_shape, n_classes):
    """Build and compile a CNN for multi-class traffic sign classification."""
    raise NotImplementedError("You need to implement this function for Task 10.")


if __name__ == '__main__':
    cnn_model = build_cnn((32, 32, 3), len(CLASS_NAMES))
    cnn_model.summary()

    n_params = cnn_model.count_params()
    print(f"\ntrainable parameters: {n_params:,}")
    assert n_params < 500_000, "model is over the 500,000 parameter budget"

## Task 11 — Train, Evaluate, and Save Your CNN

In this task, you will train your CNN model, evaluate its performance, and save it for later use. Your goal is to train your architecture from Task 10, verify its generalization performance locally, and export the trained model as a standalone file: `cnn_model.keras`.

**Input**

- `cnn_model`: the compiled model from Task 10
- `X_train`, `y_train`: training images and integer labels
- `X_val`, `y_val`: validation images and integer labels — these come from **your Task 1
  split**, so they contain no sign the model trained on
- `epochs`: `int`, default `15`
- `batch_size`: `int`, default `64`

**Output**

- A trained Keras model saved to `cnn_model.keras` in the current working directory. 

**Requirements**

- Your saved model file must be named exactly `cnn_model.keras`.
- The model must load cleanly via `keras.models.load_model("cnn_model.keras")`.
- The model outputs a probability distribution over the 10 classes for each input image.
- Total trainable parameters must stay strictly below 500,000.
- The model must achieve at least the target accuracy threshold (75%) on unseen evaluation splits.
- Do not apply validation_split during training; validation data must come strictly from the split established in Task 1 to avoid data leakage.
- You do not need to import extra functions or libraries beyond what is provided in the starter code. However, you are permitted to use standard Keras callbacks—such as `EarlyStopping` or `ModelCheckpoint`—to improve training stability or save the best model, provided they are placed under the if `__name__ == "__main__"`: block to prevent ZINC autograder import errors.
- **IMPORTANT:** Ensure the model you submit is the exact one you created in Task 10. The autograder will verify whether your model architecture matches Task 10, and it will not accept a different architecture, even if it achieves higher accuracy.

In [ ]:
if __name__ == '__main__':
    EPOCHS = 15
    BATCH_SIZE = 64

    X_train_float = X_train.astype('float32') / 255.0
    X_val_float = X_val.astype('float32') / 255.0
    X_test_float = X_test.astype('float32') / 255.0

    # Fill in this part to train your CNN using cnn_model.fit()
    history = cnn_model.fit(...)

In [ ]:
if __name__ == '__main__':
    # Staff plotting cell. Do not edit.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='train')
    axes[0].plot(history.history['val_accuracy'], label='validation')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('accuracy')
    axes[0].legend()
    axes[1].plot(history.history['loss'], label='train')
    axes[1].plot(history.history['val_loss'], label='validation')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('loss')
    axes[1].legend()
    plt.suptitle('CNN training history')
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_model(model, X, y):
    # This is the same implementation to predict and evaluate the model in the grading environment.
    probabilities = model.predict(X, verbose=0)
    predicted = np.argmax(probabilities, axis=1)
    accuracy = float(np.mean(predicted == np.asarray(y)))

    return accuracy, predicted
    

if __name__ == '__main__':
    cnn_val_accuracy, _ = evaluate_model(cnn_model, X_val_float, y_val)
    cnn_test_accuracy, cnn_test_predictions = evaluate_model(cnn_model, X_test_float, y_test)

    print(f"CNN validation accuracy: {cnn_val_accuracy:.4f}")
    print(f"CNN test accuracy      : {cnn_test_accuracy:.4f}")
    print(f"KNN test accuracy      : {mean_pool_accuracy:.4f}")

In [ ]:
if __name__ == '__main__':
    # Staff plotting cell. Do not edit.
    matrix = confusion_matrix(y_test, cnn_test_predictions)
    fig, ax = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    ax.set_title('CNN confusion matrix on the test set')
    plt.tight_layout()
    plt.show()

Look at where the mistakes are, not just how many there are.

The five non-speed-limit signs — Yield, Stop, No entry, Road work, Keep right — are
seldom confused with anything. They have distinct shapes and colours, and at 32×32
those survive.

Nearly every error is inside the block of five speed-limit signs, which are the same red
circle differing only in two small digits. At this resolution those digits are a handful
of pixels. A model that reaches around 90% overall is not making 10% of its errors evenly:
it has essentially solved half the problem and is struggling with the other half.

That is the kind of thing an accuracy number alone will never tell you, and it is what
you would take to whoever specified the camera — 32×32 is enough to know *that* there is a
speed limit, but not enough to read it.

---
## Summary

Run the cell below to collect everything you measured into one table.

In [ ]:
if __name__ == '__main__':
    # Staff summary cell. Do not edit.
    rows = [
        ("hand-crafted features + KNN", "daylight test", mean_pool_accuracy),
        ("CNN on raw pixels", "validation", cnn_val_accuracy),
        ("CNN on raw pixels", "test", cnn_test_accuracy),
    ]
    print(f"{'model':32s}{'evaluated on':16s}{'accuracy':>10s}")
    print("-" * 58)
    for model_name, split_name, value in rows:
        print(f"{model_name:32s}{split_name:16s}{value:10.4f}")

---
# Submission

Run the cell below to export your trained CNN, then download `cnn_model.keras` from the Colab Files panel.

Submission Checklist:
1. Export `comp2211_pa2.py`
2. Download `cnn_model.keras`

In [ ]:
if __name__ == '__main__':
    cnn_model.save('cnn_model.keras')
    print('saved cnn_model.keras')